# Matrix Matching Model - Local Testing Notebook

Interactive notebook for testing the **KKEdgeMatchingModel** with matrix-based matching.

Experiments covered:
1. Basic Matching (perfect match scenario)
2. Multiple Input Documents (same type)
3. Numeric Tolerance Testing
4. Mixed Document Types
5. Missing Fields Handling
6. Explainability & Top-K
7. Threshold Tuning
8. Multiple Links per Document

## Setup

In [ ]:
import json
from pprint import pprint

from nerds_nlp.models.edge.matching import (
    KKEdgeMatchingModel,
    MatchingConfig,
    MatchingContract,
    MatchResult,
)

print("Imports OK")

## Configuration

Define the matching configuration inline so experiments are self-contained.

| Field            | Strategy           | Weight | Params              |
|------------------|--------------------|--------|---------------------|
| valuation_date   | exact_match        | 0.30   |                     |
| forward_rate     | numeric_range      | 0.25   | tolerance = 0.01    |
| settlement_date  | exact_match        | 0.25   |                     |
| direction        | direction_inverse  | 0.20   |                     |

In [ ]:
CONFIG = {
    "matching": {
        "threshold": 0.85,
        "fields": [
            {"name": "valuation_date", "weight": 0.3, "strategy": "exact_match"},
            {
                "name": "forward_rate",
                "weight": 0.25,
                "strategy": "numeric_range",
                "params": {"tolerance": 0.01},
            },
            {"name": "settlement_date", "weight": 0.25, "strategy": "exact_match"},
            {"name": "direction", "weight": 0.2, "strategy": "direction_inverse"},
        ],
    }
}

print("Config loaded:")
pprint(CONFIG)

### Helper: display results

In [ ]:
def show_results(results: list[MatchResult], label: str = "") -> None:
    """Pretty-print a list of MatchResult objects."""
    if label:
        print(f"\n{'='*60}")
        print(f"  {label}")
        print(f"{'='*60}")
    for i, r in enumerate(results):
        status = "MATCHED" if r.matched else "NO MATCH"
        print(f"\n  Result [{i}] -> {status}")
        print(f"    best_candidate_document_id : {r.best_candidate_document_id}")
        print(f"    best_candidate_link_index  : {r.best_candidate_link_index}")
        print(f"    best_score                 : {r.best_score:.6f}")
        if r.explanation:
            print(f"    explanation:")
            print(f"      input_document_id : {r.explanation.input_document_id}")
            print(f"      input_link_index  : {r.explanation.input_link_index}")
            for c in r.explanation.candidates:
                print(f"      ---")
                print(f"      candidate: {c.candidate_document_id} (link {c.candidate_link_index})")
                print(f"      overall_score: {c.overall_score:.6f}")
                for fd in c.field_details:
                    print(
                        f"        {fd.field_name:20s}  "
                        f"score={fd.score:.4f}  "
                        f"weight={fd.weight:.2f}  "
                        f"contribution={fd.contribution:.4f}"
                    )
    print()

---
## Experiment 1: Basic Matching (Perfect Match)

One input link vs. two candidates. The first candidate is a perfect match
(all fields match, direction is the inverse). The second candidate has
completely different data.

In [ ]:
model = KKEdgeMatchingModel(config_dict=CONFIG, return_explainability=True)

contract_basic = {
    "input_documents": [
        {
            "id": "input-1",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "incoming",
                }
            ],
        }
    ],
    "unmatched_documents": [
        {
            "id": "cand-1",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "outgoing",
                }
            ],
        },
        {
            "id": "cand-2",
            "links": [
                {
                    "valuation_date": "2024-12-01",
                    "forward_rate": "2.0",
                    "settlement_date": "2024-12-15",
                    "direction": "incoming",
                }
            ],
        },
    ],
}

results = model.match(contract_basic)
show_results(results, "Experiment 1: Basic Matching")

assert results[0].matched is True
assert results[0].best_candidate_document_id == "cand-1"
assert results[0].best_score == 1.0
print("All assertions passed.")

---
## Experiment 2: Multiple Input Documents (Same Type)

Two input documents, each with one link. Two candidate documents.
Each input should match its corresponding candidate.

In [ ]:
model = KKEdgeMatchingModel(config_dict=CONFIG, return_explainability=True)

contract_multi_input = {
    "input_documents": [
        {
            "id": "input-1",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "incoming",
                }
            ],
        },
        {
            "id": "input-2",
            "links": [
                {
                    "valuation_date": "2024-12-01",
                    "forward_rate": "2.0",
                    "settlement_date": "2024-12-15",
                    "direction": "outgoing",
                }
            ],
        },
    ],
    "unmatched_documents": [
        {
            "id": "cand-A",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "outgoing",
                }
            ],
        },
        {
            "id": "cand-B",
            "links": [
                {
                    "valuation_date": "2024-12-01",
                    "forward_rate": "2.0",
                    "settlement_date": "2024-12-15",
                    "direction": "incoming",
                }
            ],
        },
    ],
}

results = model.match(contract_multi_input)
show_results(results, "Experiment 2: Multiple Input Documents")

assert len(results) == 2
assert results[0].matched is True
assert results[0].best_candidate_document_id == "cand-A"
assert results[1].matched is True
assert results[1].best_candidate_document_id == "cand-B"
print("All assertions passed.")

---
## Experiment 3: Numeric Tolerance Testing

Test the `numeric_range` strategy with different `forward_rate` deltas.
The configured tolerance is **0.01**, so:
- diff = 0.000 -> score = 1.00
- diff = 0.005 -> score = 0.50
- diff = 0.009 -> score = 0.10
- diff = 0.010 -> score = 0.00 (at boundary)
- diff = 0.020 -> score = 0.00 (beyond)

In [ ]:
model = KKEdgeMatchingModel(config_dict=CONFIG, return_explainability=True)

base_rate = 1.2345
deltas = [0.000, 0.002, 0.005, 0.008, 0.009, 0.010, 0.020]

candidates = []
for i, delta in enumerate(deltas):
    candidates.append(
        {
            "id": f"cand-delta-{delta}",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": str(base_rate + delta),
                    "settlement_date": "2025-03-15",
                    "direction": "outgoing",
                }
            ],
        }
    )

contract_tolerance = {
    "input_documents": [
        {
            "id": "input-1",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": str(base_rate),
                    "settlement_date": "2025-03-15",
                    "direction": "incoming",
                }
            ],
        }
    ],
    "unmatched_documents": candidates,
}

results = model.match(contract_tolerance)
show_results(results, "Experiment 3: Numeric Tolerance Testing")

# Show per-candidate forward_rate scores
print("Per-candidate forward_rate scores:")
print(f"  {'Delta':>8s}  {'Rate':>10s}  {'FwdRate Score':>14s}  {'Overall':>10s}")
print(f"  {'-'*8}  {'-'*10}  {'-'*14}  {'-'*10}")
for c in results[0].explanation.candidates:
    fwd_detail = next(fd for fd in c.field_details if fd.field_name == "forward_rate")
    delta_val = float(c.candidate_document_id.split("delta-")[1])
    print(
        f"  {delta_val:>8.3f}  "
        f"{base_rate + delta_val:>10.4f}  "
        f"{fwd_detail.score:>14.4f}  "
        f"{c.overall_score:>10.4f}"
    )

# Best match should be the exact match (delta=0)
assert results[0].best_candidate_document_id == "cand-delta-0.0"
assert results[0].best_score == 1.0
print("\nAll assertions passed.")

---
## Experiment 4: Mixed Document Types

Candidates with varying field combinations:
- Candidate A: perfect match
- Candidate B: same dates but wrong direction (not inverse)
- Candidate C: close forward_rate, different dates
- Candidate D: completely different

In [ ]:
model = KKEdgeMatchingModel(config_dict=CONFIG, return_explainability=True)

contract_mixed = {
    "input_documents": [
        {
            "id": "input-1",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "incoming",
                }
            ],
        }
    ],
    "unmatched_documents": [
        {
            "id": "cand-A-perfect",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "outgoing",
                }
            ],
        },
        {
            "id": "cand-B-wrong-dir",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "incoming",
                }
            ],
        },
        {
            "id": "cand-C-close-rate",
            "links": [
                {
                    "valuation_date": "2025-02-01",
                    "forward_rate": "1.2340",
                    "settlement_date": "2025-04-01",
                    "direction": "outgoing",
                }
            ],
        },
        {
            "id": "cand-D-no-match",
            "links": [
                {
                    "valuation_date": "2024-06-01",
                    "forward_rate": "5.0",
                    "settlement_date": "2024-07-01",
                    "direction": "sell",
                }
            ],
        },
    ],
}

results = model.match(contract_mixed)
show_results(results, "Experiment 4: Mixed Document Types")

# Summary table
print("Score Summary:")
print(f"  {'Candidate':>20s}  {'Score':>8s}")
print(f"  {'-'*20}  {'-'*8}")
for c in results[0].explanation.candidates:
    print(f"  {c.candidate_document_id:>20s}  {c.overall_score:>8.4f}")

assert results[0].best_candidate_document_id == "cand-A-perfect"
assert results[0].best_score == 1.0
print("\nAll assertions passed.")

---
## Experiment 5: Missing Fields Handling

Test what happens when:
- **Input** is missing a field -> that field is excluded from scoring entirely
- **Candidate** is missing a field -> candidate gets score=0 for that field

In [ ]:
model = KKEdgeMatchingModel(config_dict=CONFIG, return_explainability=True)

# Case A: Input missing "forward_rate" — only 3 fields are scored
contract_missing_input = {
    "input_documents": [
        {
            "id": "input-no-rate",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    # forward_rate intentionally omitted
                    "settlement_date": "2025-03-15",
                    "direction": "incoming",
                }
            ],
        }
    ],
    "unmatched_documents": [
        {
            "id": "cand-1",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "outgoing",
                }
            ],
        }
    ],
}

results_a = model.match(contract_missing_input)
show_results(results_a, "Experiment 5a: Input Missing forward_rate")

# forward_rate is excluded; remaining 3 fields all match -> score = 1.0
assert results_a[0].matched is True
assert results_a[0].best_score == 1.0
# Explanation should only have 3 fields, not 4
field_names = [fd.field_name for fd in results_a[0].explanation.candidates[0].field_details]
assert "forward_rate" not in field_names
assert len(field_names) == 3
print("Case A assertions passed.\n")

# Case B: Candidate missing "settlement_date" — candidate gets 0 for that field
contract_missing_cand = {
    "input_documents": [
        {
            "id": "input-1",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "incoming",
                }
            ],
        }
    ],
    "unmatched_documents": [
        {
            "id": "cand-no-settle",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    # settlement_date intentionally omitted
                    "direction": "outgoing",
                }
            ],
        }
    ],
}

results_b = model.match(contract_missing_cand)
show_results(results_b, "Experiment 5b: Candidate Missing settlement_date")

# settlement_date contributes 0; other 3 fields match perfectly
# expected score = (0.3 + 0.25 + 0.0 + 0.2) / 1.0 = 0.75
assert results_b[0].best_score < 1.0
expected_score = (0.3 * 1.0 + 0.25 * 1.0 + 0.25 * 0.0 + 0.2 * 1.0) / (0.3 + 0.25 + 0.25 + 0.2)
assert abs(results_b[0].best_score - expected_score) < 1e-10
print(f"Expected score = {expected_score:.4f}, got {results_b[0].best_score:.4f}")
print("Case B assertions passed.")

---
## Experiment 6: Explainability & Top-K

Test the explainability output and the `top_k_explainability` parameter.

With 4 candidates and `top_k_explainability=2`, only the top 2 scored
candidates should appear in the explanation.

In [ ]:
model_full = KKEdgeMatchingModel(config_dict=CONFIG, return_explainability=True)
model_top2 = KKEdgeMatchingModel(
    config_dict=CONFIG, return_explainability=True, top_k_explainability=2
)

contract_explain = {
    "input_documents": [
        {
            "id": "input-1",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "incoming",
                }
            ],
        }
    ],
    "unmatched_documents": [
        {
            "id": "cand-best",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "outgoing",
                }
            ],
        },
        {
            "id": "cand-second",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2340",
                    "settlement_date": "2025-03-15",
                    "direction": "outgoing",
                }
            ],
        },
        {
            "id": "cand-third",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2300",
                    "settlement_date": "2025-04-01",
                    "direction": "outgoing",
                }
            ],
        },
        {
            "id": "cand-worst",
            "links": [
                {
                    "valuation_date": "2024-06-01",
                    "forward_rate": "9.99",
                    "settlement_date": "2024-07-01",
                    "direction": "incoming",
                }
            ],
        },
    ],
}

results_full = model_full.match(contract_explain)
results_top2 = model_top2.match(contract_explain)

show_results(results_full, "Experiment 6a: Full Explainability (all candidates)")

# Full explainability should have all 4 candidates
assert len(results_full[0].explanation.candidates) == 4
print(f"Full explanation: {len(results_full[0].explanation.candidates)} candidates\n")

show_results(results_top2, "Experiment 6b: Top-2 Explainability")

# Top-K should have only 2
assert len(results_top2[0].explanation.candidates) == 2
# Top candidate should be the best match
assert results_top2[0].explanation.candidates[0].candidate_document_id == "cand-best"
print(f"Top-2 explanation: {len(results_top2[0].explanation.candidates)} candidates")
print("All assertions passed.")

---
## Experiment 7: Threshold Tuning

Show how adjusting the threshold affects which candidates count as "matched".

We use a candidate that scores ~0.80 and test thresholds of 0.70, 0.80, 0.85, 0.90.

In [ ]:
contract_threshold = {
    "input_documents": [
        {
            "id": "input-1",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "incoming",
                }
            ],
        }
    ],
    "unmatched_documents": [
        {
            "id": "cand-partial",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-20",  # different settlement
                    "direction": "outgoing",
                }
            ],
        }
    ],
}

thresholds = [0.50, 0.70, 0.75, 0.80, 0.85, 0.90]

print("Experiment 7: Threshold Tuning")
print(f"  {'Threshold':>10s}  {'Score':>8s}  {'Matched':>8s}")
print(f"  {'-'*10}  {'-'*8}  {'-'*8}")

for t in thresholds:
    cfg = {
        "matching": {
            "threshold": t,
            "fields": CONFIG["matching"]["fields"],
        }
    }
    m = KKEdgeMatchingModel(config_dict=cfg)
    res = m.match(contract_threshold)
    print(f"  {t:>10.2f}  {res[0].best_score:>8.4f}  {str(res[0].matched):>8s}")

print("\nDone.")

---
## Experiment 8: Multiple Links per Document

One input document with 2 links, each scored independently against the
candidate pool.

In [ ]:
model = KKEdgeMatchingModel(config_dict=CONFIG, return_explainability=True)

contract_multi_links = {
    "input_documents": [
        {
            "id": "input-1",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "incoming",
                },
                {
                    "valuation_date": "2024-06-01",
                    "forward_rate": "3.0",
                    "settlement_date": "2024-06-15",
                    "direction": "buy",
                },
            ],
        }
    ],
    "unmatched_documents": [
        {
            "id": "cand-A",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "outgoing",
                }
            ],
        },
        {
            "id": "cand-B",
            "links": [
                {
                    "valuation_date": "2024-06-01",
                    "forward_rate": "3.0",
                    "settlement_date": "2024-06-15",
                    "direction": "sell",
                }
            ],
        },
    ],
}

results = model.match(contract_multi_links)
show_results(results, "Experiment 8: Multiple Links per Document")

assert len(results) == 2  # One result per input link
# Link 0 -> matches cand-A
assert results[0].matched is True
assert results[0].best_candidate_document_id == "cand-A"
assert results[0].best_score == 1.0
# Link 1 -> matches cand-B (buy/sell inverse)
assert results[1].matched is True
assert results[1].best_candidate_document_id == "cand-B"
assert results[1].best_score == 1.0
print("All assertions passed.")

---
## Experiment 9: JSON Serialization

Verify that results (including explainability) serialize cleanly to JSON.

In [ ]:
model = KKEdgeMatchingModel(config_dict=CONFIG, return_explainability=True)
results = model.match(contract_basic)

result_json = results[0].model_dump()
print("Experiment 9: JSON Serialization")
print(json.dumps(result_json, indent=2))

assert isinstance(result_json, dict)
assert "best_candidate_document_id" in result_json
assert "explanation" in result_json
assert result_json["explanation"] is not None

# Round-trip: JSON string -> dict -> verify
json_str = json.dumps(result_json)
roundtrip = json.loads(json_str)
assert roundtrip == result_json
print("\nJSON round-trip passed.")

---
## Experiment 10: Load Config from YAML File

Load the default `config/matching_config.yaml` and verify it works.

In [ ]:
# Uses the default YAML config path
model_yaml = KKEdgeMatchingModel(return_explainability=True)

print("Experiment 10: YAML Config")
print(f"  Threshold : {model_yaml.config.threshold}")
print(f"  Fields    : {len(model_yaml.config.fields)}")
for fc in model_yaml.config.fields:
    print(f"    - {fc.name:20s}  strategy={fc.strategy:20s}  weight={fc.weight}  params={fc.params}")

results = model_yaml.match(contract_basic)
show_results(results, "YAML Config Results")

assert results[0].matched is True
assert results[0].best_candidate_document_id == "cand-1"
print("All assertions passed.")

---
## Experiment 11: Custom Config - Different Weights

Show how changing field weights affects the final ranking.

We use a scenario where candidate A has correct dates but wrong rate,
and candidate B has correct rate but wrong dates. By adjusting weights
we can flip which candidate wins.

In [ ]:
contract_weights = {
    "input_documents": [
        {
            "id": "input-1",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "1.2345",
                    "settlement_date": "2025-03-15",
                    "direction": "incoming",
                }
            ],
        }
    ],
    "unmatched_documents": [
        {
            "id": "cand-dates-ok",
            "links": [
                {
                    "valuation_date": "2025-01-15",
                    "forward_rate": "9.9999",  # way off
                    "settlement_date": "2025-03-15",
                    "direction": "outgoing",
                }
            ],
        },
        {
            "id": "cand-rate-ok",
            "links": [
                {
                    "valuation_date": "2024-01-01",  # wrong date
                    "forward_rate": "1.2345",
                    "settlement_date": "2024-02-01",  # wrong date
                    "direction": "outgoing",
                }
            ],
        },
    ],
}

# Config A: heavy weight on dates
config_dates_heavy = {
    "matching": {
        "threshold": 0.0,
        "fields": [
            {"name": "valuation_date", "weight": 0.40, "strategy": "exact_match"},
            {"name": "forward_rate", "weight": 0.05, "strategy": "numeric_range", "params": {"tolerance": 0.01}},
            {"name": "settlement_date", "weight": 0.40, "strategy": "exact_match"},
            {"name": "direction", "weight": 0.15, "strategy": "direction_inverse"},
        ],
    }
}

# Config B: heavy weight on forward_rate
config_rate_heavy = {
    "matching": {
        "threshold": 0.0,
        "fields": [
            {"name": "valuation_date", "weight": 0.05, "strategy": "exact_match"},
            {"name": "forward_rate", "weight": 0.70, "strategy": "numeric_range", "params": {"tolerance": 0.01}},
            {"name": "settlement_date", "weight": 0.05, "strategy": "exact_match"},
            {"name": "direction", "weight": 0.20, "strategy": "direction_inverse"},
        ],
    }
}

model_a = KKEdgeMatchingModel(config_dict=config_dates_heavy, return_explainability=True)
model_b = KKEdgeMatchingModel(config_dict=config_rate_heavy, return_explainability=True)

res_a = model_a.match(contract_weights)
res_b = model_b.match(contract_weights)

print("Experiment 11: Custom Config - Different Weights\n")
print("Config A (dates heavy):")
print(f"  Winner: {res_a[0].best_candidate_document_id} (score={res_a[0].best_score:.4f})")
for c in res_a[0].explanation.candidates:
    print(f"    {c.candidate_document_id}: {c.overall_score:.4f}")

print("\nConfig B (rate heavy):")
print(f"  Winner: {res_b[0].best_candidate_document_id} (score={res_b[0].best_score:.4f})")
for c in res_b[0].explanation.candidates:
    print(f"    {c.candidate_document_id}: {c.overall_score:.4f}")

# When dates are heavy, dates-ok candidate wins
assert res_a[0].best_candidate_document_id == "cand-dates-ok"
# When rate is heavy, rate-ok candidate wins
assert res_b[0].best_candidate_document_id == "cand-rate-ok"
print("\nAll assertions passed.")

---
## Summary

All experiments demonstrate the matrix matching engine working correctly:

| # | Experiment                     | Status |
|---|--------------------------------|--------|
| 1 | Basic Matching                 | Passed |
| 2 | Multiple Input Documents       | Passed |
| 3 | Numeric Tolerance              | Passed |
| 4 | Mixed Document Types           | Passed |
| 5 | Missing Fields                 | Passed |
| 6 | Explainability & Top-K         | Passed |
| 7 | Threshold Tuning               | Passed |
| 8 | Multiple Links per Document    | Passed |
| 9 | JSON Serialization             | Passed |
| 10| YAML Config Loading            | Passed |
| 11| Custom Weights                 | Passed |